# Lab 04–05: Spark SQL DataFrames & Window Analytics

**Z5008 Big Data Lab (MTech Core) · IIT Madras Zanzibar**

Partitioned window ranking and cumulative aggregations in PySpark with SQLite fallback.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/inyalala/iitmz-teaching-site/blob/main/courses/z5008/labs/BD02_Spark_SQL_DataFrames_Tuning.ipynb)

Run cells top to bottom (`Runtime > Run all`). Spark steps include pure-Python fallbacks when executed outside a Spark/JVM container.

## 1. Spark SQL Window Analytics (`RANK()` & Cumulative `SUM()`)

Using `SparkSession` in Colab/Docker or SQLite window functions locally produces identical partitioned rankings and running totals.

In [1]:
import sqlite3

orders = [
    ("T1", "Zanzibar", 100.0, "2026-05-01T09:00"),
    ("T2", "Zanzibar", 250.0, "2026-05-01T10:00"),
    ("T3", "Pemba", 180.0, "2026-05-01T09:30"),
    ("T4", "Pemba", 60.0, "2026-05-01T11:00"),
]

try:
    from pyspark.sql import SparkSession, Window
    import pyspark.sql.functions as F
    spark = SparkSession.builder.master("local[2]").appName("Z5008-BD02").getOrCreate()
    spark.sparkContext.setLogLevel("ERROR")
    df = spark.createDataFrame(orders, ["tx_id", "region", "amount", "ts"])
    w_r = Window.partitionBy("region").orderBy(F.desc("amount"))
    w_s = Window.partitionBy("region").orderBy("ts").rowsBetween(Window.unboundedPreceding, Window.currentRow)
    out = [tuple(r) for r in df.withColumn("rnk", F.rank().over(w_r)).withColumn("cum", F.sum("amount").over(w_s)).orderBy("region", "rnk").collect()]
    engine = "PySpark"
except Exception:
    conn = sqlite3.connect(":memory:")
    conn.execute("CREATE TABLE orders (tx_id TEXT, region TEXT, amount REAL, ts TEXT)")
    conn.executemany("INSERT INTO orders VALUES (?, ?, ?, ?)", orders)
    out = conn.execute("""
        SELECT tx_id, region, amount, ts,
               RANK() OVER (PARTITION BY region ORDER BY amount DESC) AS rnk,
               SUM(amount) OVER (PARTITION BY region ORDER BY ts ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cum
        FROM orders ORDER BY region, rnk
    """).fetchall()
    engine = "SQLite Window fallback"

print("Engine:", engine)
for row in out:
    print(" ", row)
assert out[0][0] == "T3" and out[0][4] == 1


Engine: SQLite Window fallback
  ('T3', 'Pemba', 180.0, '2026-05-01T09:30', 1, 180.0)
  ('T4', 'Pemba', 60.0, '2026-05-01T11:00', 2, 240.0)
  ('T2', 'Zanzibar', 250.0, '2026-05-01T10:00', 1, 350.0)
  ('T1', 'Zanzibar', 100.0, '2026-05-01T09:00', 2, 100.0)
